# Predicting Late Deliveries at Approval Time: Olist Brazilian E-Commerce

McGill INSY 674, Fall 2026, final group project.

**Team:** Fidan Shiraliyeva, Hasan Malik, Jay Malhotra, Jingyu Wu, Mia Scalabrin, Peace Isaac, Tasnim Hassan (student IDs to add)

**Research question.** Using only information available when an order is approved, how accurately can we predict which orders will arrive after the promised delivery date, and which factors contribute most to that risk?

Sections follow the Machine Learning Project Checklist (Géron, Appendix B) used in the rubric. Locked decisions are in `DECISIONS.md`; team rules in `CONVENTIONS.md`.

## 0. Setup

In [1]:
# Make the repo root importable so shared code in src/ is used everywhere.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import config
from src.data import load_raw, build_population, time_split

pd.set_option("display.float_format", "{:,.4f}".format)
np.random.seed(config.RANDOM_STATE)

## 1. Frame the Problem and Look at the Big Picture

Full framing (objective, users, current workaround, framing choices, metrics, baselines) is in README section 1. The cells below produce every number quoted there.

### 1.1 Population and target

Only delivered orders have a label. The censoring rule drops orders promised in the last 30 days before the data snapshot, because their late deliveries would not have been recorded yet (DECISIONS.md D4).

In [2]:
orders = load_raw("orders")
population, funnel = build_population(orders)
funnel

,step,orders
0,Raw orders,99441
1,Status 'delivered',96478
2,Has approval and delivery dates,96456
3,Delivery not before purchase,96456
4,Purchased from 2017-01-01,96189
5,Promised on or before 2018-08-01 (censoring rule),85371


In [3]:
print(f"Late orders: {population[config.TARGET].sum():,} of {len(population):,} "
      f"({population[config.TARGET].mean():.1%})")

Late orders: 5,986 of 85,371 (7.0%)


### 1.1a Approved order-level features
Attach the pre-aggregated feature table using `order_id`. The project population and calendar-date target remain authoritative. Reviews and post-delivery columns are never included in `X`.

In [ ]:
from src.integration import integrate_features, model_features
population = integrate_features(population)
print(f"Integrated: {len(population):,} orders, {len(model_features(population).columns)} candidate features")
assert population.order_id.is_unique


### 1.2 Why it matters: late orders and review scores

Reviews are used here only to size the business problem. They are written after approval, so they are never features (DECISIONS.md D5).

In [4]:
reviews = load_raw("reviews")
# Some orders have several reviews; average them so each order counts once.
order_review = reviews.groupby("order_id")["review_score"].mean()
with_review = population.join(order_review, on="order_id")

review_summary = with_review.groupby(config.TARGET)["review_score"].agg(
    mean_stars="mean",
    share_1_or_2_stars=lambda s: (s <= 2).mean(),
    orders_with_review="count",
)
review_summary.index = review_summary.index.map({0: "on time", 1: "late"})
review_summary

,mean_stars,share_1_or_2_stars,orders_with_review
is_late,,,
on time,4.2814,0.0930,78929
late,2.2104,0.6251,5848


### 1.3 Current workaround: padded promises

In [5]:
promised_days = (population["order_estimated_delivery_date"]
                 - population["order_purchase_timestamp"].dt.normalize()).dt.days
actual_days = (population["order_delivered_customer_date"]
               - population["order_purchase_timestamp"]).dt.days
print(f"Median promised window: {promised_days.median():.0f} days")
print(f"Median actual delivery: {actual_days.median():.0f} days")

Median promised window: 24 days
Median actual delivery: 10 days


In [6]:
monthly = (population.set_index("order_purchase_timestamp")[config.TARGET]
           .resample("ME").agg(["size", "mean"])
           .rename(columns={"size": "orders", "mean": "late_rate"}))
monthly

,orders,late_rate
order_purchase_timestamp,,
2017-01-31,748,0.0294
2017-02-28,1641,0.0299
2017-03-31,2546,0.0456
2017-04-30,2303,0.0656
2017-05-31,3545,0.0299
2017-06-30,3135,0.0303
2017-07-31,3872,0.0279
2017-08-31,4193,0.0291
2017-09-30,4150,0.0439


### 1.4 Train/test split and baseline B0

Time-based split: train on the past, test on the future (DECISIONS.md D6). The test set is not used again until section 5.3.

In [7]:
train, test = time_split(population)
split_summary = pd.DataFrame({
    "first_purchase": [train["order_purchase_timestamp"].min(), test["order_purchase_timestamp"].min()],
    "last_purchase": [train["order_purchase_timestamp"].max(), test["order_purchase_timestamp"].max()],
    "orders": [len(train), len(test)],
    "late": [train[config.TARGET].sum(), test[config.TARGET].sum()],
    "late_rate": [train[config.TARGET].mean(), test[config.TARGET].mean()],
}, index=["train", "test"])
split_summary

,first_purchase,last_purchase,orders,late,late_rate
train,2017-01-05 11:56:06,2018-03-31 23:54:10,64038,5110,0.0798
test,2018-04-01 00:11:32,2018-07-27 22:40:35,21333,876,0.0411


In [8]:
# B0 (no skill): PR-AUC of a random ranking equals the late rate.
# Accuracy of predicting 'on time' for everything shows why accuracy is not used.
print(f"B0 test PR-AUC: {test[config.TARGET].mean():.3f}")
print(f"Accuracy of always predicting on time (test): {1 - test[config.TARGET].mean():.1%}")

B0 test PR-AUC: 0.041
Accuracy of always predicting on time (test): 95.9%


## Where the rest of the project lives

This notebook covers section 1 and produces every number quoted in README section 1. The other sections are in their own notebooks, in this order:

| Section | Notebook | Owner |
|---|---|---|
| 2. Explore the Data | `02_explore_the_data.ipynb` | Hasan Malik |
| 3. Prepare the Data | `03_prepare_the_data.ipynb` | Mia Scalabrin |
| 4.1, 4.5 Model comparison and shortlist | `04a_compare_models.ipynb` | Fidan Shiraliyeva |
| 4.2, 4.3 Interpretation and error analysis | `04b_interpretation_and_errors.ipynb` | Jingyu Wu |
| 4.4 Feature iteration and split check | `04c_feature_iteration.ipynb` | Jay Malhotra |
| 5. Fine-Tune the System | `05_fine_tune.ipynb` | Tasnim Hassan |

Sections 6 and 7 (conclusion, assumptions and limitations) are in the README.